# Notebook 02 — Preprocesamiento, LST e índices espectrales

Este notebook genera productos anuales crudos derivados de Landsat 8/9 para el flujo de modelado de LST/SUHI en Barranquilla, Colombia.

## Objetivo

Aplicar enmascaramiento de calidad, extraer la temperatura superficial terrestre, calcular índices espectrales derivados de Landsat y exportar composiciones anuales medianas para el periodo operativo 2013–2025.

## Insumos principales

- Productos Landsat 8/9 Collection 2 Level-2 disponibles en Google Earth Engine.
- WRS Path 9 / Row 52.
- Grilla canónica de referencia almacenada externamente en Google Drive.

## Salidas principales

El notebook exporta productos GeoTIFF anuales hacia una carpeta temporal de Google Drive y luego consolida las salidas verificadas en:

    03_annual_raw_composites/v3_expanded_raw/

Productos esperados:

    13 años × 9 productos = 117 archivos GeoTIFF

Las variables exportadas son:

- NDVI
- NDMI
- NDBI
- UI
- SAVI
- BSI
- LST
- n_obs_indices
- n_obs_LST

## Política del repositorio

Los archivos GeoTIFF generados son productos pesados externos y no deben subirse a GitHub.

El notebook sí se incluye en el repositorio porque documenta el procedimiento metodológico y permite la reproducibilidad del flujo de preprocesamiento.


## Módulo 01 — Instalación, importación y autenticación

Este módulo instala las dependencias necesarias, monta Google Drive e inicializa Google Earth Engine.

Si Earth Engine no está autenticado, el notebook solicitará autorización.


In [1]:
# ============================================================
# MÓDULO 01 — INSTALACIÓN, IMPORTACIÓN Y AUTENTICACIÓN
# ============================================================

!pip -q install earthengine-api rasterio pyproj pandas

from pathlib import Path
import shutil
import pandas as pd

import ee
import rasterio
from pyproj import Transformer
from google.colab import drive

# Montaje estándar de Google Drive.
drive.mount("/content/drive")

# Proyecto de Google Cloud usado para inicializar Earth Engine.
# Edite este valor si su entorno usa otro proyecto.
PROJECT_ID = "inundaciones-msc"

try:
    ee.Initialize(project=PROJECT_ID)
    print("Earth Engine ya estaba inicializado.")
except Exception as e:
    print("Inicialización falló:")
    print(e)
    print("\nAutenticando Earth Engine en modo notebook...")
    ee.Authenticate(auth_mode="notebook")
    ee.Initialize(project=PROJECT_ID)
    print("Earth Engine inicializado correctamente.")

print(ee.String("Earth Engine OK").getInfo())


Mounted at /content/drive
Inicialización falló:
Please authorize access to your Earth Engine account by running

earthengine authenticate

in your command line, or ee.Authenticate() in Python, and then retry.

Autenticando Earth Engine en modo notebook...
To authorize access needed by Earth Engine, open the following URL in a web browser and follow the instructions. If the web browser does not start automatically, please manually browse the URL below.

    https://code.earthengine.google.com/client-auth?scopes=https%3A//www.googleapis.com/auth/earthengine%20https%3A//www.googleapis.com/auth/cloud-platform%20https%3A//www.googleapis.com/auth/drive%20https%3A//www.googleapis.com/auth/devstorage.full_control&request_id=nmM8bLU9U7kFCRUpVLdquJru2FemZgMc1cPjW2ygmX0&tc=K5KJIn3ZOxuyiZzoZf_T6DkiAjZK5SVVwXWBHfuqe6Y&cc=qRe3RpPGHCNdrmf8L_JtFQYK9KbZrY3Oe0pUoVOunBE

The authorization workflow will generate a code, which you should paste in the box below.
Enter verification code: 4/1AdkVLPyNPLw0TLKQz

## Módulo 02 — Configuración de rutas y búsqueda de la grilla de referencia

Este módulo identifica la carpeta externa del proyecto y localiza automáticamente el raster de referencia de la grilla canónica.

La ruta prioritaria se toma de los notebooks base ya validados:

    01_aoi_reference/reference_grid/model_domain_reference.tif

También se aceptan rutas heredadas como:

    01_aoi_reference/canonical_grid_reference.tif
    02_reference/canonical_grid_reference.tif

El notebook procesa únicamente 2013–2025, aunque la carpeta histórica del proyecto conserve el nombre `2013_2026`.


In [2]:
# ============================================================
# MÓDULO 02 — RUTAS DEL PROYECTO Y GRILLA DE REFERENCIA
# ============================================================

# Candidatos principales para la raíz externa del proyecto.
# Se prioriza 2013_2025, pero se mantiene compatibilidad con la carpeta histórica 2013_2026.
PROJECT_CANDIDATES = [
    Path("/content/drive/MyDrive/Barranquilla_LST_2013_2025_Modeling"),
    Path("/content/drive/MyDrive/Barranquilla_LST_2013_2026_Modeling"),
]

PROJECT_DIR = None
for candidate in PROJECT_CANDIDATES:
    if candidate.exists():
        PROJECT_DIR = candidate
        break

if PROJECT_DIR is None:
    raise FileNotFoundError(
        "No se encontró ninguna carpeta externa del proyecto.\n"
        "Rutas revisadas:\n"
        + "\n".join([str(p) for p in PROJECT_CANDIDATES])
    )

print("PROJECT_DIR seleccionado:", PROJECT_DIR)
print("Existe:", PROJECT_DIR.exists())

# Rutas prioritarias detectadas en los notebooks base del proyecto.
REFERENCE_CANDIDATES = [
    PROJECT_DIR / "01_aoi_reference" / "reference_grid" / "model_domain_reference.tif",
    PROJECT_DIR / "01_aoi_reference" / "canonical_grid_reference.tif",
    PROJECT_DIR / "02_reference" / "canonical_grid_reference.tif",
]

REF_RASTER = None
for candidate in REFERENCE_CANDIDATES:
    if candidate.exists():
        REF_RASTER = candidate
        break

# Búsqueda de respaldo si ninguna ruta prioritaria existe.
if REF_RASTER is None:
    search_names = ["model_domain_reference.tif", "canonical_grid_reference.tif"]
    matches = []
    for name in search_names:
        matches.extend(PROJECT_DIR.rglob(name))
    matches = sorted(set(matches))

    print("\nNo se encontró la grilla en las rutas prioritarias.")
    print("Búsqueda de respaldo dentro de PROJECT_DIR:")
    for m in matches:
        print("  ", m)

    if matches:
        # Preferencia por model_domain_reference dentro de reference_grid.
        preferred = [m for m in matches if "reference_grid" in str(m) and m.name == "model_domain_reference.tif"]
        REF_RASTER = preferred[0] if preferred else matches[0]

if REF_RASTER is None or not REF_RASTER.exists():
    raise FileNotFoundError(
        "No se encontró el raster de referencia de grilla canónica.\n\n"
        "Rutas prioritarias revisadas:\n"
        + "\n".join([str(p) for p in REFERENCE_CANDIDATES])
        + "\n\nCopie el raster de referencia aprobado a una de estas rutas o revise PROJECT_DIR."
    )

print("\nREF_RASTER seleccionado:", REF_RASTER)
print("Existe:", REF_RASTER.exists())

# Carpetas de salida externas.
V3_CONSOLIDATED_DIR = PROJECT_DIR / "03_annual_raw_composites" / "v3_expanded_raw"
V3_CONSOLIDATED_DIR.mkdir(parents=True, exist_ok=True)

# Carpeta temporal usada por Earth Engine Export.image.toDrive.
# Earth Engine recibe solo el nombre de carpeta, no una ruta completa.
EXPORT_FOLDER = "Barranquilla_LST_2013_2025_02_V3_STAGING"
V3_STAGING_DIR = Path("/content/drive/MyDrive") / EXPORT_FOLDER

print("\nCarpeta STAGING esperada:", V3_STAGING_DIR)
print("Carpeta consolidada V3:", V3_CONSOLIDATED_DIR)


PROJECT_DIR seleccionado: /content/drive/MyDrive/Barranquilla_LST_2013_2026_Modeling
Existe: True

REF_RASTER seleccionado: /content/drive/MyDrive/Barranquilla_LST_2013_2026_Modeling/01_aoi_reference/reference_grid/model_domain_reference.tif
Existe: True

Carpeta STAGING esperada: /content/drive/MyDrive/Barranquilla_LST_2013_2025_02_V3_STAGING
Carpeta consolidada V3: /content/drive/MyDrive/Barranquilla_LST_2013_2026_Modeling/03_annual_raw_composites/v3_expanded_raw


## Módulo 03 — Lectura de la grilla canónica

Este módulo lee el raster de referencia y extrae la geometría necesaria para exportar todos los productos con la misma grilla espacial.


In [3]:
# ============================================================
# MÓDULO 03 — LECTURA DE GRILLA CANÓNICA
# ============================================================

with rasterio.open(REF_RASTER) as src:
    ref_bounds = src.bounds
    ref_crs = src.crs
    ref_transform = src.transform
    ref_width = src.width
    ref_height = src.height

print("Raster referencia:", REF_RASTER)
print("CRS:", ref_crs)
print("Shape:", ref_height, ref_width)
print("Transform:", ref_transform)
print("Bounds:", ref_bounds)

transformer = Transformer.from_crs(ref_crs, "EPSG:4326", always_xy=True)

lon_min, lat_min = transformer.transform(ref_bounds.left, ref_bounds.bottom)
lon_max, lat_max = transformer.transform(ref_bounds.right, ref_bounds.top)

REGION_EE = ee.Geometry.Rectangle(
    [lon_min, lat_min, lon_max, lat_max],
    proj="EPSG:4326",
    geodesic=False
)

CRS_OUT = str(ref_crs)
SCALE = abs(ref_transform.a)

CRS_TRANSFORM = [
    ref_transform.a,
    ref_transform.b,
    ref_transform.c,
    ref_transform.d,
    ref_transform.e,
    ref_transform.f,
]

print("Región EE lon/lat:", [lon_min, lat_min, lon_max, lat_max])
print("CRS salida:", CRS_OUT)
print("Escala nominal:", SCALE)
print("CRS transform export:", CRS_TRANSFORM)


Raster referencia: /content/drive/MyDrive/Barranquilla_LST_2013_2026_Modeling/01_aoi_reference/reference_grid/model_domain_reference.tif
CRS: EPSG:32618
Shape: 1115 1257
Transform: | 30.00, 0.00, 495735.00|
| 0.00,-30.00, 1230045.00|
| 0.00, 0.00, 1.00|
Bounds: BoundingBox(left=495735.0, bottom=1196595.0, right=533445.0, top=1230045.0)
Región EE lon/lat: [-75.03901822361303, 10.824683592539946, -74.69371946525051, 11.127052518330629]
CRS salida: EPSG:32618
Escala nominal: 30.0
CRS transform export: [30.0, 0.0, 495735.0, 0.0, -30.0, 1230045.0]


## Módulo 04 — Parámetros Landsat 8/9 y periodo operativo

Este módulo fija el periodo 2013–2025, Path/Row 9/52, variables, umbrales radiométricos y criterios mínimos de observación.


In [4]:
# ============================================================
# MÓDULO 04 — PARÁMETROS LANDSAT 8/9 2013–2025
# ============================================================

START_YEAR = 2013
END_YEAR = 2025
YEARS_TO_EXPORT = list(range(START_YEAR, END_YEAR + 1))

WRS_PATH = 9
WRS_ROW = 52

# Mínimo de observaciones válidas para aceptar el valor anual mediano.
MIN_OBS_ANNUAL = 2

# Buffer hídrico equivalente a 1 píxel Landsat.
WATER_BUFFER_M = 30

# QA
MASK_CIRRUS_HARD = False

# Variables espectrales y térmica.
SPECTRAL_VARS = ["NDVI", "NDMI", "NDBI", "UI", "SAVI", "BSI"]
ALL_EXPORT_VARS = SPECTRAL_VARS + ["LST_C"]
EXPECTED_PRODUCTS = ["NDVI", "NDMI", "NDBI", "UI", "SAVI", "BSI", "LST", "n_obs_indices", "n_obs_LST"]

# Parámetros de máscara hídrica robusta.
NDVI_MAX_WATER = 0.10
CLAMP_MIN = -0.20
CLAMP_MAX = 0.20
GREEN_MIN = 0.03
NIR_MIN = 0.02
NBINS_OTSU = 256
SAMPLE_MAX = 2_000_000
MIN_COMP_PX = 9

# Filtro físico LST en grados Celsius.
LST_MIN_C = 15.0
LST_MAX_C = 80.0

# Reflectancia física escalada.
RHO_MIN = 0.0
RHO_MAX = 1.20

# Denominadores mínimos para índices.
DEN_ND_MIN = 0.05
DEN_BSI_MIN = 0.10
DEN_SAVI_MIN = 0.05

SR_BANDS = ["SR_B2", "SR_B3", "SR_B4", "SR_B5", "SR_B6", "SR_B7"]

print("Periodo operativo:", START_YEAR, "-", END_YEAR)
print("Años:", YEARS_TO_EXPORT)
print("WRS Path/Row:", WRS_PATH, WRS_ROW)
print("Variables espectrales:", SPECTRAL_VARS)
print("Productos esperados:", len(YEARS_TO_EXPORT), "años ×", len(EXPECTED_PRODUCTS), "productos =", len(YEARS_TO_EXPORT) * len(EXPECTED_PRODUCTS), "GeoTIFF")


Periodo operativo: 2013 - 2025
Años: [2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025]
WRS Path/Row: 9 52
Variables espectrales: ['NDVI', 'NDMI', 'NDBI', 'UI', 'SAVI', 'BSI']
Productos esperados: 13 años × 9 productos = 117 GeoTIFF


## Módulo 05 — Funciones auxiliares de QA, Otsu e índices

Este módulo define las funciones generales usadas para QA_PIXEL, índices de razón, umbral Otsu y control de denominadores.


In [5]:
# ============================================================
# MÓDULO 05 — FUNCIONES AUXILIARES
# ============================================================

def bit_zero(img, band, bit):
    return img.select(band).bitwiseAnd(1 << bit).eq(0)


def bit_one(img, band, bit):
    return img.select(band).bitwiseAnd(1 << bit).neq(0)


def ratio_index(num, den, name, den_min=0.05, lo=-1.0, hi=1.0):
    """
    Índice de razón con denominador mínimo y rango físico.
    Los valores fuera de rango quedan enmascarados.
    """
    den_ok = den.abs().gt(den_min)
    idx = num.divide(den).rename(name)
    idx = idx.updateMask(den_ok)
    idx = idx.updateMask(idx.gte(lo).And(idx.lte(hi)))
    return idx


def ratio_raw_for_water(num, den, name, den_min=0.01):
    """
    Índice auxiliar para clasificación de agua.
    Es menos restrictivo que los índices finales.
    """
    den_ok = den.abs().gt(den_min)
    return num.divide(den).rename(name).updateMask(den_ok)


def otsu_threshold_ee(index_img, mask_img, region, scale=30, nbins=256):
    """
    Calcula umbral Otsu en Earth Engine sobre index_img dentro de mask_img.
    Devuelve ee.Number. Si no hay histograma, devuelve 0.0.
    """
    band_name = ee.String(index_img.bandNames().get(0))

    hist_raw = (
        index_img.updateMask(mask_img)
        .reduceRegion(
            reducer=ee.Reducer.histogram(
                maxBuckets=nbins,
                minBucketWidth=0.001
            ),
            geometry=region,
            scale=scale,
            bestEffort=True,
            maxPixels=SAMPLE_MAX,
            tileScale=4
        )
        .get(band_name)
    )

    def compute_from_hist(h):
        h = ee.Dictionary(h)
        counts = ee.Array(h.get("histogram"))
        means = ee.Array(h.get("bucketMeans"))

        total = counts.reduce(ee.Reducer.sum(), [0]).get([0])
        sum_total = counts.multiply(means).reduce(ee.Reducer.sum(), [0]).get([0])
        mean_total = sum_total.divide(total)

        size = means.length().get([0])
        indices = ee.List.sequence(1, size.subtract(1))

        def compute_bss(i):
            i = ee.Number(i)
            counts1 = counts.slice(0, 0, i)
            means1 = means.slice(0, 0, i)

            weight1 = counts1.reduce(ee.Reducer.sum(), [0]).get([0])
            sum1 = counts1.multiply(means1).reduce(ee.Reducer.sum(), [0]).get([0])

            weight2 = total.subtract(weight1)
            sum2 = sum_total.subtract(sum1)

            mean1 = ee.Number(
                ee.Algorithms.If(
                    ee.Number(weight1).gt(0),
                    ee.Number(sum1).divide(weight1),
                    0
                )
            )

            mean2 = ee.Number(
                ee.Algorithms.If(
                    ee.Number(weight2).gt(0),
                    ee.Number(sum2).divide(weight2),
                    0
                )
            )

            bss = (
                ee.Number(weight1).multiply(mean1.subtract(mean_total).pow(2))
                .add(ee.Number(weight2).multiply(mean2.subtract(mean_total).pow(2)))
            )
            return bss

        bss = indices.map(compute_bss)
        maxval = ee.Number(ee.List(bss).reduce(ee.Reducer.max()))
        idx = ee.Number(bss.indexOf(maxval)).add(1)
        threshold = means.toList().get(idx)
        return ee.Number(threshold)

    threshold = ee.Number(
        ee.Algorithms.If(
            hist_raw,
            compute_from_hist(hist_raw),
            0.0
        )
    )
    return threshold

print("Funciones auxiliares definidas correctamente.")


Funciones auxiliares definidas correctamente.


## Módulo 06 — Procesamiento por escena Landsat 8/9

Este módulo aplica QA_PIXEL, escala reflectancia y LST, calcula índices espectrales y genera máscaras válidas por escena.


In [6]:
# ============================================================
# MÓDULO 06 — PROCESAMIENTO POR ESCENA LANDSAT 8/9
# ============================================================

def add_scene_products_l89(img):
    """
    Procesamiento por escena:
    - QA_PIXEL
    - valid_SR / valid_LST
    - escalado físico
    - índices finales filtrados
    - máscara hídrica robusta
    - máscara terrestre estricta
    """

    # QA_PIXEL
    fill_ok = bit_zero(img, "QA_PIXEL", 0)
    dilated_ok = bit_zero(img, "QA_PIXEL", 1)
    cirrus_ok = bit_zero(img, "QA_PIXEL", 2)
    cloud_ok = bit_zero(img, "QA_PIXEL", 3)
    cloud_shadow_ok = bit_zero(img, "QA_PIXEL", 4)
    snow_ok = bit_zero(img, "QA_PIXEL", 5)

    hard_qa = (
        fill_ok
        .And(dilated_ok)
        .And(cloud_ok)
        .And(cloud_shadow_ok)
        .And(snow_ok)
    )

    if MASK_CIRRUS_HARD:
        hard_qa = hard_qa.And(cirrus_ok)

    radsat_ok = img.select("QA_RADSAT").eq(0)

    # Escalado físico C2 L2
    optical = img.select(SR_BANDS).multiply(0.0000275).add(-0.2)

    blue = optical.select("SR_B2")
    green = optical.select("SR_B3")
    red = optical.select("SR_B4")
    nir = optical.select("SR_B5")
    swir1 = optical.select("SR_B6")
    swir2 = optical.select("SR_B7")

    lst_c = (
        img.select("ST_B10")
        .multiply(0.00341802)
        .add(149.0)
        .subtract(273.15)
        .rename("LST_C")
    )

    # Validez radiométrica
    valid_sr_dn = img.select(SR_BANDS).reduce(ee.Reducer.min()).gt(0)

    valid_reflectance = (
        blue.gt(RHO_MIN).And(blue.lt(RHO_MAX))
        .And(green.gt(RHO_MIN)).And(green.lt(RHO_MAX))
        .And(red.gt(RHO_MIN)).And(red.lt(RHO_MAX))
        .And(nir.gt(RHO_MIN)).And(nir.lt(RHO_MAX))
        .And(swir1.gt(RHO_MIN)).And(swir1.lt(RHO_MAX))
        .And(swir2.gt(RHO_MIN)).And(swir2.lt(RHO_MAX))
    )

    valid_lst_dn = img.select("ST_B10").gt(0)
    valid_lst_phys = lst_c.gt(LST_MIN_C).And(lst_c.lt(LST_MAX_C))
    valid_lst = valid_lst_dn.And(valid_lst_phys)

    hard_obs = (
        hard_qa
        .And(valid_sr_dn)
        .And(radsat_ok)
        .And(valid_lst)
    )

    hard_final_vars = hard_obs.And(valid_reflectance)

    # Índices auxiliares para agua
    ndvi_water = ratio_raw_for_water(
        nir.subtract(red),
        nir.add(red),
        "NDVI_water",
        den_min=0.01
    )

    mndwi = ratio_raw_for_water(
        green.subtract(swir1),
        green.add(swir1),
        "MNDWI",
        den_min=0.01
    )

    # Máscara hídrica robusta
    qa_water = bit_one(img, "QA_PIXEL", 7)

    t_otsu = otsu_threshold_ee(
        index_img=mndwi,
        mask_img=hard_obs,
        region=REGION_EE,
        scale=SCALE,
        nbins=NBINS_OTSU
    )

    t_w = ee.Number(t_otsu).max(CLAMP_MIN).min(CLAMP_MAX)
    anti_shadow = green.gt(GREEN_MIN).Or(nir.gt(NIR_MIN))

    water_spectral = (
        mndwi.gt(t_w)
        .And(ndvi_water.lt(NDVI_MAX_WATER))
        .And(anti_shadow)
        .And(hard_obs)
    )

    water_pre = (
        qa_water
        .Or(water_spectral)
        .And(hard_obs)
        .rename("water_pre")
    )

    water_open = (
        water_pre
        .focal_min(radius=1, units="pixels")
        .focal_max(radius=1, units="pixels")
    )

    water_close = (
        water_open
        .focal_max(radius=2, units="pixels")
        .focal_min(radius=2, units="pixels")
    )

    connected = water_close.connectedPixelCount(
        maxSize=100,
        eightConnected=True
    )

    water_mask = (
        water_close
        .And(connected.gte(MIN_COMP_PX))
        .And(hard_obs)
        .rename("water_mask")
    )

    water_buffer = water_mask.focal_max(
        radius=WATER_BUFFER_M,
        units="meters"
    )

    land_mask = hard_obs.And(water_mask.Not()).rename("land_mask")
    land_mask_strict_obs = hard_obs.And(water_buffer.Not()).rename("land_mask_strict_obs")

    land_mask_strict_vars = (
        hard_final_vars
        .And(water_buffer.Not())
        .rename("land_mask_strict")
    )

    # Índices finales filtrados
    ndvi = ratio_index(nir.subtract(red), nir.add(red), "NDVI", den_min=DEN_ND_MIN, lo=-1.0, hi=1.0)
    ndmi = ratio_index(nir.subtract(swir1), nir.add(swir1), "NDMI", den_min=DEN_ND_MIN, lo=-1.0, hi=1.0)
    ndbi = ratio_index(swir1.subtract(nir), swir1.add(nir), "NDBI", den_min=DEN_ND_MIN, lo=-1.0, hi=1.0)
    ui = ratio_index(swir2.subtract(nir), swir2.add(nir), "UI", den_min=DEN_ND_MIN, lo=-1.0, hi=1.0)

    savi_den = nir.add(red).add(0.5)
    savi = ratio_index(nir.subtract(red).multiply(1.5), savi_den, "SAVI", den_min=DEN_SAVI_MIN, lo=-1.5, hi=1.5)

    bsi_den = swir1.add(red).add(nir).add(blue)
    bsi = ratio_index(
        swir1.add(red).subtract(nir.add(blue)),
        bsi_den,
        "BSI",
        den_min=DEN_BSI_MIN,
        lo=-1.0,
        hi=1.0
    )

    variables = (
        ee.Image.cat([ndvi, ndmi, ndbi, ui, savi, bsi, lst_c])
        .updateMask(land_mask_strict_vars)
    )

    valid_obs = land_mask_strict_vars.rename("valid_obs").uint8()
    date = ee.Date(img.get("system:time_start"))

    out = (
        variables
        .addBands(valid_obs)
        .addBands(hard_obs.rename("hard_obs").uint8())
        .addBands(hard_final_vars.rename("hard_final_vars").uint8())
        .addBands(water_mask.uint8())
        .addBands(land_mask.uint8())
        .addBands(land_mask_strict_obs.uint8())
        .addBands(land_mask_strict_vars.uint8())
        .set("system:time_start", img.get("system:time_start"))
        .set("date", date.format("YYYY-MM-dd"))
        .set("year", date.get("year"))
        .set("month", date.get("month"))
        .set("doy", date.getRelative("day", "year").add(1))
        .set("LANDSAT_PRODUCT_ID", img.get("LANDSAT_PRODUCT_ID"))
        .set("CLOUD_COVER", img.get("CLOUD_COVER"))
        .set("CLOUD_COVER_LAND", img.get("CLOUD_COVER_LAND"))
        .set("SPACECRAFT_ID", img.get("SPACECRAFT_ID"))
        .set("otsu_mndwi", t_otsu)
        .set("water_threshold_used", t_w)
    )

    return out

print("Función add_scene_products_l89 definida correctamente.")


Función add_scene_products_l89 definida correctamente.


## Módulo 07 — Construcción de la colección Landsat 8/9

Este módulo consulta las colecciones Landsat 8 y Landsat 9 Collection 2 Level-2 en Google Earth Engine, filtra Path/Row 9/52 y aplica el procesamiento por escena.


In [7]:
# ============================================================
# MÓDULO 07 — COLECCIÓN LANDSAT 8/9 PROCESADA
# ============================================================

def prepare_collection(collection_id, sensor_name):
    return (
        ee.ImageCollection(collection_id)
        .filterDate(f"{START_YEAR}-01-01", f"{END_YEAR + 1}-01-01")
        .filterMetadata("WRS_PATH", "equals", WRS_PATH)
        .filterMetadata("WRS_ROW", "equals", WRS_ROW)
        .map(lambda img: img.set("SENSOR", sensor_name))
    )

l8 = prepare_collection("LANDSAT/LC08/C02/T1_L2", "LC08")
l9 = prepare_collection("LANDSAT/LC09/C02/T1_L2", "LC09")

l89_raw = l8.merge(l9).sort("system:time_start")
print("Escenas brutas L8/L9:", l89_raw.size().getInfo())

l89_processed = l89_raw.map(add_scene_products_l89)
print("Escenas procesadas:", l89_processed.size().getInfo())

# Resumen anual de escenas disponibles.
year_rows = []
for year in YEARS_TO_EXPORT:
    col_y = l89_raw.filterDate(f"{year}-01-01", f"{year + 1}-01-01")
    year_rows.append({
        "year": year,
        "n_scenes_raw": col_y.size().getInfo(),
    })

df_scene_counts = pd.DataFrame(year_rows)
display(df_scene_counts)


Escenas brutas L8/L9: 364
Escenas procesadas: 364


,year,n_scenes_raw
0,2013,14
1,2014,21
2,2015,23
3,2016,23
4,2017,21
5,2018,22
6,2019,22
7,2020,22
8,2021,25
9,2022,42


## Módulo 08 — Funciones de composiciones anuales

Este módulo define las funciones para generar composiciones medianas anuales y conteos mínimos de observaciones válidas.


In [8]:
# ============================================================
# MÓDULO 08 — COMPOSICIONES ANUALES
# ============================================================

def annual_collection(year):
    start = ee.Date.fromYMD(year, 1, 1)
    end = start.advance(1, "year")
    return l89_processed.filterDate(start, end)


def annual_median_var(year, varname):
    col_y = annual_collection(year)
    count = col_y.select(varname).count().rename(f"n_obs_{varname}")
    med = col_y.select(varname).median().rename(f"{varname}_median_{year}")
    med = med.updateMask(count.gte(MIN_OBS_ANNUAL))

    return med.set({
        "year": year,
        "variable": varname,
        "min_obs_required": MIN_OBS_ANNUAL,
        "n_scenes_year": col_y.size()
    })


def annual_n_obs_indices(year):
    col_y = annual_collection(year)
    counts = ee.Image.cat([
        col_y.select(v).count().rename(f"n_{v}") for v in SPECTRAL_VARS
    ])

    n_min = (
        counts
        .reduce(ee.Reducer.min())
        .rename(f"n_obs_indices_{year}")
        .uint16()
    )

    return n_min.set({
        "year": year,
        "variable": "n_obs_indices",
        "n_scenes_year": col_y.size()
    })


def annual_n_obs_lst(year):
    col_y = annual_collection(year)
    n = (
        col_y.select("LST_C")
        .count()
        .rename(f"n_obs_LST_{year}")
        .uint16()
    )

    return n.set({
        "year": year,
        "variable": "n_obs_LST",
        "n_scenes_year": col_y.size()
    })

print("Funciones de composición anual definidas correctamente.")


Funciones de composición anual definidas correctamente.


## Módulo 09 — Lanzamiento de exportaciones a Google Drive

Este módulo lanza las exportaciones anuales a la carpeta temporal STAGING de Google Drive.

**Advertencia:** esta celda lanza 117 tareas de Earth Engine. Ejecútela solo cuando los módulos anteriores estén verificados.


In [9]:
# ============================================================
# MÓDULO 09 — EXPORTACIÓN A GOOGLE DRIVE
# ============================================================

RUN_EXPORTS = True  # Cambie a False si solo desea revisar el notebook sin lanzar tareas.


def export_image_to_drive(image, description, filename_prefix):
    task = ee.batch.Export.image.toDrive(
        image=image.clip(REGION_EE),
        description=description,
        folder=EXPORT_FOLDER,
        fileNamePrefix=filename_prefix,
        region=REGION_EE,
        crs=CRS_OUT,
        crsTransform=CRS_TRANSFORM,
        maxPixels=1e13,
        fileFormat="GeoTIFF"
    )
    task.start()
    print("Export iniciado:", description)


if RUN_EXPORTS:
    total_tasks = 0

    for year in YEARS_TO_EXPORT:
        col_y = annual_collection(year)
        n_scenes = col_y.size().getInfo()
        dates = col_y.aggregate_array("date").getInfo()

        print("\n================================================")
        print(f"AÑO {year}")
        print("Escenas:", n_scenes)
        print("Fechas:", dates)
        print("================================================")

        for varname in ALL_EXPORT_VARS:
            img = annual_median_var(year, varname)

            if varname == "LST_C":
                out_name = f"LST_median_{year}"
            else:
                out_name = f"{varname}_median_{year}"

            export_image_to_drive(
                image=img,
                description=f"EXP10Bv3_{out_name}",
                filename_prefix=f"EXP10Bv3_{out_name}"
            )
            total_tasks += 1

        img_nidx = annual_n_obs_indices(year)
        export_image_to_drive(
            image=img_nidx,
            description=f"EXP10Bv3_n_obs_indices_{year}",
            filename_prefix=f"EXP10Bv3_n_obs_indices_{year}"
        )
        total_tasks += 1

        img_nlst = annual_n_obs_lst(year)
        export_image_to_drive(
            image=img_nlst,
            description=f"EXP10Bv3_n_obs_LST_{year}",
            filename_prefix=f"EXP10Bv3_n_obs_LST_{year}"
        )
        total_tasks += 1

    print("\nExportaciones lanzadas:", total_tasks)
    print("Productos esperados:", len(YEARS_TO_EXPORT) * len(EXPECTED_PRODUCTS))
    print("Revise las tareas en Earth Engine Tasks.")
    print("Carpeta Drive STAGING de salida:", EXPORT_FOLDER)
else:
    print("RUN_EXPORTS = False. No se lanzaron tareas de Earth Engine.")



AÑO 2013
Escenas: 14
Fechas: ['2013-04-01', '2013-05-13', '2013-06-14', '2013-06-30', '2013-07-16', '2013-08-01', '2013-08-17', '2013-09-02', '2013-09-18', '2013-10-20', '2013-11-05', '2013-11-21', '2013-12-07', '2013-12-23']
Export iniciado: EXP10Bv3_NDVI_median_2013
Export iniciado: EXP10Bv3_NDMI_median_2013
Export iniciado: EXP10Bv3_NDBI_median_2013
Export iniciado: EXP10Bv3_UI_median_2013
Export iniciado: EXP10Bv3_SAVI_median_2013
Export iniciado: EXP10Bv3_BSI_median_2013
Export iniciado: EXP10Bv3_LST_median_2013
Export iniciado: EXP10Bv3_n_obs_indices_2013
Export iniciado: EXP10Bv3_n_obs_LST_2013

AÑO 2014
Escenas: 21
Fechas: ['2014-01-24', '2014-02-09', '2014-02-25', '2014-03-13', '2014-03-29', '2014-04-14', '2014-05-16', '2014-06-01', '2014-06-17', '2014-07-03', '2014-07-19', '2014-08-04', '2014-08-20', '2014-09-05', '2014-09-21', '2014-10-07', '2014-10-23', '2014-11-08', '2014-11-24', '2014-12-10', '2014-12-26']
Export iniciado: EXP10Bv3_NDVI_median_2014
Export iniciado: EXP10

## Módulo 10 — Verificación de exportaciones en STAGING

Ejecute este módulo después de que terminen todas las tareas de Earth Engine.

El resultado esperado es:

    117 archivos GeoTIFF


In [12]:
# ============================================================
# MÓDULO 10 — VERIFICACIÓN DE EXPORTACIONES EN STAGING
# ============================================================

expected = []
for y in YEARS_TO_EXPORT:
    for v in EXPECTED_PRODUCTS:
        if v == "LST":
            expected.append(f"EXP10Bv3_LST_median_{y}.tif")
        elif v == "n_obs_indices":
            expected.append(f"EXP10Bv3_n_obs_indices_{y}.tif")
        elif v == "n_obs_LST":
            expected.append(f"EXP10Bv3_n_obs_LST_{y}.tif")
        else:
            expected.append(f"EXP10Bv3_{v}_median_{y}.tif")

found = sorted([p.name for p in V3_STAGING_DIR.glob("EXP10Bv3_*.tif")]) if V3_STAGING_DIR.exists() else []
missing = [f for f in expected if f not in found]
extra = [f for f in found if f not in expected]

print("Carpeta STAGING:", V3_STAGING_DIR)
print("Existe:", V3_STAGING_DIR.exists())
print("Esperados:", len(expected))
print("Encontrados:", len(found))
print("Faltantes:", len(missing))
print("Extras:", len(extra))

if missing:
    print("\nFALTANTES:")
    for f in missing:
        print("  ", f)
else:
    print("\nV3 2013–2025 completo en STAGING.")

if extra:
    print("\nARCHIVOS EXTRA:")
    for f in extra:
        print("  ", f)

rows = []
for y in YEARS_TO_EXPORT:
    year_expected = [f for f in expected if f"_{y}.tif" in f]
    year_found = [f for f in year_expected if f in found]
    rows.append({
        "year": y,
        "expected": len(year_expected),
        "found": len(year_found),
        "complete": len(year_found) == len(year_expected),
    })

df_year = pd.DataFrame(rows)
display(df_year)


Carpeta STAGING: /content/drive/MyDrive/Barranquilla_LST_2013_2025_02_V3_STAGING
Existe: True
Esperados: 117
Encontrados: 0
Faltantes: 117
Extras: 0

FALTANTES:
   EXP10Bv3_NDVI_median_2013.tif
   EXP10Bv3_NDMI_median_2013.tif
   EXP10Bv3_NDBI_median_2013.tif
   EXP10Bv3_UI_median_2013.tif
   EXP10Bv3_SAVI_median_2013.tif
   EXP10Bv3_BSI_median_2013.tif
   EXP10Bv3_LST_median_2013.tif
   EXP10Bv3_n_obs_indices_2013.tif
   EXP10Bv3_n_obs_LST_2013.tif
   EXP10Bv3_NDVI_median_2014.tif
   EXP10Bv3_NDMI_median_2014.tif
   EXP10Bv3_NDBI_median_2014.tif
   EXP10Bv3_UI_median_2014.tif
   EXP10Bv3_SAVI_median_2014.tif
   EXP10Bv3_BSI_median_2014.tif
   EXP10Bv3_LST_median_2014.tif
   EXP10Bv3_n_obs_indices_2014.tif
   EXP10Bv3_n_obs_LST_2014.tif
   EXP10Bv3_NDVI_median_2015.tif
   EXP10Bv3_NDMI_median_2015.tif
   EXP10Bv3_NDBI_median_2015.tif
   EXP10Bv3_UI_median_2015.tif
   EXP10Bv3_SAVI_median_2015.tif
   EXP10Bv3_BSI_median_2015.tif
   EXP10Bv3_LST_median_2015.tif
   EXP10Bv3_n_obs_indices_

,year,expected,found,complete
0,2013,9,0,False
1,2014,9,0,False
2,2015,9,0,False
3,2016,9,0,False
4,2017,9,0,False
5,2018,9,0,False
6,2019,9,0,False
7,2020,9,0,False
8,2021,9,0,False
9,2022,9,0,False


## Módulo 11 — Copia a carpeta consolidada del proyecto

Ejecute este módulo solo si la verificación anterior muestra 117/117 archivos y cero faltantes.


In [16]:
# ============================================================
# MÓDULO 11 — COPIA A CARPETA CONSOLIDADA V3
# ============================================================

from pathlib import Path
import shutil
import pandas as pd

print("=" * 80)
print("MÓDULO 11 — COPIA A CARPETA CONSOLIDADA V3")
print("=" * 80)

# ------------------------------------------------------------
# 11.0. Parámetros autónomos del módulo
# ------------------------------------------------------------

START_YEAR = 2013
END_YEAR = 2025
YEARS = list(range(START_YEAR, END_YEAR + 1))

SPECTRAL_VARS = ["NDVI", "NDMI", "NDBI", "UI", "SAVI", "BSI"]
EXPORT_PREFIX = "EXP10Bv3"

EXPECTED_TOTAL = len(YEARS) * 9

print(f"Periodo operativo: {START_YEAR}–{END_YEAR}")
print(f"Variables espectrales: {SPECTRAL_VARS}")
print(f"Prefijo esperado: {EXPORT_PREFIX}")
print(f"Total esperado: {EXPECTED_TOTAL} GeoTIFF")

# ------------------------------------------------------------
# 11.1. Resolver PROJECT_DIR si no existe en memoria
# ------------------------------------------------------------

if "PROJECT_DIR" not in globals():
    candidate_project_dirs = [
        Path("/content/drive/MyDrive/Barranquilla_LST_2013_2025_Modeling"),
        Path("/content/drive/MyDrive/Barranquilla_LST_2013_2026_Modeling"),
    ]

    PROJECT_DIR = None

    for p in candidate_project_dirs:
        if p.exists():
            PROJECT_DIR = p
            break

    if PROJECT_DIR is None:
        raise FileNotFoundError(
            "No se encontró la carpeta del proyecto en las rutas esperadas:\n"
            + "\n".join(str(p) for p in candidate_project_dirs)
        )

print(f"\nPROJECT_DIR:")
print(PROJECT_DIR)

# ------------------------------------------------------------
# 11.2. Carpeta consolidada final
# ------------------------------------------------------------

V3_DIR = PROJECT_DIR / "03_annual_raw_composites" / "v3_expanded_raw"
V3_DIR.mkdir(parents=True, exist_ok=True)

print(f"\nCarpeta consolidada V3:")
print(V3_DIR)

# ------------------------------------------------------------
# 11.3. Definición de productos esperados
# ------------------------------------------------------------

EXPECTED_PRODUCTS = []

for year in YEARS:
    for var in SPECTRAL_VARS:
        EXPECTED_PRODUCTS.append(f"{EXPORT_PREFIX}_{var}_median_{year}.tif")

    EXPECTED_PRODUCTS.append(f"{EXPORT_PREFIX}_LST_median_{year}.tif")
    EXPECTED_PRODUCTS.append(f"{EXPORT_PREFIX}_n_obs_indices_{year}.tif")
    EXPECTED_PRODUCTS.append(f"{EXPORT_PREFIX}_n_obs_LST_{year}.tif")

print(f"\nProductos esperados: {len(EXPECTED_PRODUCTS)}")

# ------------------------------------------------------------
# 11.4. Búsqueda robusta de archivos exportados en Google Drive
# ------------------------------------------------------------

drive_root = Path("/content/drive/MyDrive")

candidate_dirs = []

# Carpeta explícita de staging, si existe en memoria
if "STAGING_DIR" in globals():
    candidate_dirs.append(Path(STAGING_DIR))

# Carpetas probables por nombre
probable_names = [
    "GEE_exports",
    "gee_exports",
    "GEE_EXPORTS",
    "EarthEngine",
    "earthengine",
    "02_preprocessing_lst_indices_staging",
    "02_preprocessing_staging",
    "LST_exports",
    "landsat_exports",
    "EXP10Bv3",
]

for name in probable_names:
    p = drive_root / name
    if p.exists():
        candidate_dirs.append(p)

# Buscar carpetas candidatas dentro de MyDrive
for p in drive_root.glob("*"):
    if p.is_dir():
        pname = p.name.lower()
        if (
            "export" in pname
            or "gee" in pname
            or "earth" in pname
            or "landsat" in pname
            or "barranquilla" in pname
            or "lst" in pname
            or "exp10" in pname
        ):
            candidate_dirs.append(p)

# Incluir PROJECT_DIR completo por si las exportaciones quedaron dentro del proyecto
candidate_dirs.append(PROJECT_DIR)

# Eliminar duplicados preservando orden
unique_candidate_dirs = []
seen = set()

for p in candidate_dirs:
    p = Path(p)
    if p.exists() and str(p) not in seen:
        unique_candidate_dirs.append(p)
        seen.add(str(p))

print("\nCarpetas candidatas revisadas:")
for p in unique_candidate_dirs:
    print(f" - {p}")

if len(unique_candidate_dirs) == 0:
    raise RuntimeError(
        "No se encontraron carpetas candidatas en Google Drive. "
        "Verifique dónde quedaron las exportaciones de Earth Engine."
    )

# ------------------------------------------------------------
# 11.5. Buscar productos esperados en las carpetas candidatas
# ------------------------------------------------------------

found_files = {}

for folder in unique_candidate_dirs:
    for fname in EXPECTED_PRODUCTS:
        if fname in found_files:
            continue

        direct = folder / fname

        if direct.exists():
            found_files[fname] = direct
            continue

        matches = list(folder.rglob(fname))
        if len(matches) > 0:
            found_files[fname] = matches[0]

print(f"\nArchivos encontrados con nombres esperados: {len(found_files)} / {len(EXPECTED_PRODUCTS)}")

if len(found_files) > 0:
    print("\nEjemplos de archivos encontrados:")
    for i, (fname, path) in enumerate(found_files.items()):
        if i >= 10:
            break
        print(f" - {fname}")
        print(f"   {path}")

missing = [fname for fname in EXPECTED_PRODUCTS if fname not in found_files]

# ------------------------------------------------------------
# 11.6. Diagnóstico si no encontró nombres exactos
# ------------------------------------------------------------

if len(found_files) == 0:
    print("\nNo se encontraron archivos con los nombres exactos esperados.")
    print("Se listan GeoTIFF candidatos para diagnosticar carpeta o prefijo real:")

    candidate_tifs = []

    for folder in unique_candidate_dirs:
        candidate_tifs.extend(list(folder.rglob("*.tif")))
        candidate_tifs.extend(list(folder.rglob("*.tiff")))

    candidate_tifs = sorted(candidate_tifs, key=lambda p: str(p))

    print(f"\nTotal GeoTIFF encontrados en carpetas candidatas: {len(candidate_tifs)}")

    for p in candidate_tifs[:80]:
        print(f" - {p}")

    raise RuntimeError(
        "No se encontraron productos con los nombres exactos esperados. "
        "Revise los ejemplos impresos arriba para identificar el prefijo o carpeta real."
    )

# ------------------------------------------------------------
# 11.7. Control estricto de completitud
# ------------------------------------------------------------

if missing:
    print("\nArchivos faltantes:")
    for fname in missing[:80]:
        print(f" - {fname}")

    if len(missing) > 80:
        print(f" ... y {len(missing) - 80} más")

    raise RuntimeError(
        f"No se copia porque faltan {len(missing)} archivos V3. "
        "Revise si las tareas de Earth Engine terminaron, si el prefijo cambió "
        "o si Google Drive guardó los archivos en otra carpeta."
    )

print("\nTodos los productos esperados fueron localizados.")

# ------------------------------------------------------------
# 11.8. Copia a carpeta consolidada V3
# ------------------------------------------------------------

copied_records = []

for fname in EXPECTED_PRODUCTS:
    src = Path(found_files[fname])
    dst = V3_DIR / fname

    if src.resolve() == dst.resolve():
        action = "already_in_place"
    else:
        shutil.copy2(src, dst)
        action = "copied"

    copied_records.append({
        "filename": fname,
        "source_path": str(src),
        "destination_path": str(dst),
        "action": action,
        "size_mb": dst.stat().st_size / (1024 ** 2)
    })

copy_df = pd.DataFrame(copied_records)

print(f"\nProductos registrados en V3: {len(copy_df)}")
print(f"Tamaño total V3: {copy_df['size_mb'].sum():.2f} MB")
print("\nAcciones:")
print(copy_df["action"].value_counts())

display(copy_df.head())

# ------------------------------------------------------------
# 11.9. Guardar reporte de copia
# ------------------------------------------------------------

QC_DIR = PROJECT_DIR / "00_documentation" / "quality_control" / "02_preprocessing_lst_indices_2013_2025"
QC_DIR.mkdir(parents=True, exist_ok=True)

copy_report_path = QC_DIR / "02_v3_copy_report_2013_2025.csv"
copy_df.to_csv(copy_report_path, index=False, encoding="utf-8-sig")

print(f"\nReporte de copia guardado en:")
print(copy_report_path)

print("\nMÓDULO 11 completado correctamente.")

MÓDULO 11 — COPIA A CARPETA CONSOLIDADA V3
Periodo operativo: 2013–2025
Variables espectrales: ['NDVI', 'NDMI', 'NDBI', 'UI', 'SAVI', 'BSI']
Prefijo esperado: EXP10Bv3
Total esperado: 117 GeoTIFF

PROJECT_DIR:
/content/drive/MyDrive/Barranquilla_LST_2013_2026_Modeling

Carpeta consolidada V3:
/content/drive/MyDrive/Barranquilla_LST_2013_2026_Modeling/03_annual_raw_composites/v3_expanded_raw

Productos esperados: 117

Carpetas candidatas revisadas:
 - /content/drive/MyDrive/Barranquilla
 - /content/drive/MyDrive/GEE_Exports
 - /content/drive/MyDrive/GEE_inundaciones
 - /content/drive/MyDrive/GEE_susceptibilidad_inundacion
 - /content/drive/MyDrive/Barranquilla_LST_2013_2026_Modeling
 - /content/drive/MyDrive/Barranquilla_LST_2013_2026_10B_V3_STAGING
 - /content/drive/MyDrive/Barranquilla_climate_forcings_42
 - /content/drive/MyDrive/Barranquilla_LST_2013_2025_02_V3_STAGING (1)
 - /content/drive/MyDrive/Barranquilla_LST_2013_2025_02_V3_STAGING

Archivos encontrados con nombres esperados:

,filename,source_path,destination_path,action,size_mb
0,EXP10Bv3_NDVI_median_2013.tif,/content/drive/MyDrive/Barranquilla_LST_2013_2...,/content/drive/MyDrive/Barranquilla_LST_2013_2...,already_in_place,8.744957
1,EXP10Bv3_NDMI_median_2013.tif,/content/drive/MyDrive/Barranquilla_LST_2013_2...,/content/drive/MyDrive/Barranquilla_LST_2013_2...,already_in_place,8.905208
2,EXP10Bv3_NDBI_median_2013.tif,/content/drive/MyDrive/Barranquilla_LST_2013_2...,/content/drive/MyDrive/Barranquilla_LST_2013_2...,already_in_place,8.904315
3,EXP10Bv3_UI_median_2013.tif,/content/drive/MyDrive/Barranquilla_LST_2013_2...,/content/drive/MyDrive/Barranquilla_LST_2013_2...,already_in_place,8.796737
4,EXP10Bv3_SAVI_median_2013.tif,/content/drive/MyDrive/Barranquilla_LST_2013_2...,/content/drive/MyDrive/Barranquilla_LST_2013_2...,already_in_place,8.775608



Reporte de copia guardado en:
/content/drive/MyDrive/Barranquilla_LST_2013_2026_Modeling/00_documentation/quality_control/02_preprocessing_lst_indices_2013_2025/02_v3_copy_report_2013_2025.csv

MÓDULO 11 completado correctamente.


## Módulo 12 — Resumen final del módulo 02

Este módulo resume la disponibilidad final de los productos V3 consolidados.


In [17]:
# ============================================================
# MÓDULO 12 — RESUMEN FINAL
# ============================================================

files = sorted(V3_CONSOLIDATED_DIR.glob("EXP10Bv3_*.tif"))

print("Destino consolidado:", V3_CONSOLIDATED_DIR)
print("Archivos V3 consolidados:", len(files))
print("Esperados:", len(YEARS_TO_EXPORT) * len(EXPECTED_PRODUCTS))

for f in files[:10]:
    print(f.name)

if len(files) == len(YEARS_TO_EXPORT) * len(EXPECTED_PRODUCTS):
    print("\nV3 consolidado correctamente para 2013–2025.")
else:
    print("\nRevisión necesaria: el número de archivos consolidados no coincide con lo esperado.")


Destino consolidado: /content/drive/MyDrive/Barranquilla_LST_2013_2026_Modeling/03_annual_raw_composites/v3_expanded_raw
Archivos V3 consolidados: 126
Esperados: 1521
EXP10Bv3_BSI_median_2013.tif
EXP10Bv3_BSI_median_2014.tif
EXP10Bv3_BSI_median_2015.tif
EXP10Bv3_BSI_median_2016.tif
EXP10Bv3_BSI_median_2017.tif
EXP10Bv3_BSI_median_2018.tif
EXP10Bv3_BSI_median_2019.tif
EXP10Bv3_BSI_median_2020.tif
EXP10Bv3_BSI_median_2021.tif
EXP10Bv3_BSI_median_2022.tif

Revisión necesaria: el número de archivos consolidados no coincide con lo esperado.
